# TensorFlow Datasets 

![Status](https://img.shields.io/static/v1.svg?label=Status&message=Finished&color=brightgreen)
[![Source](https://img.shields.io/static/v1.svg?label=GitHub&message=Source&color=181717&logo=GitHub)](https://github.com/particle1331/ok-transformer/blob/master/docs/nb/tf/00-datasets.ipynb)
[![Stars](https://img.shields.io/github/stars/particle1331/ok-transformer?style=social)](https://github.com/particle1331/ok-transformer)

---

**References:**  {cite}`RaschkaMirjalili2019`

## Introduction

To train neural networks, we usually work with data that is too large to fit in memory. In this case, it would not be possible to simply use `fit` on Keras models and we have to load the data from storage in chunks. TensorFlow provides the `tf.data.Dataset` API to facilitate efficient pipelines which support batching, shuffling, and mapping for loading and lazy preprocessing of input data.

<br>

In [ ]:
import random
import warnings
import numpy as np 
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

import tensorflow as tf
import tensorflow.keras as kr
import tensorflow_datasets as tfds

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

warnings.simplefilter(action="once")
backend_inline.set_matplotlib_formats('svg')

print(tf.__version__)
print(tf.config.list_physical_devices())

## Dataset from tensors

We can initialize a TF dataset from an existing tensor as follows:

In [ ]:
a = tf.range(10)
ds = tf.data.Dataset.from_tensor_slices(a)
print(ds)

In [ ]:
for item in ds:
    print(item)

In [ ]:
ds_batch = ds.batch(4, drop_remainder=False) # Analogous to drop_last in PyTorch
for batch in ds_batch:
    print(batch)

To create joint datasets, simply pass a tuple of tensors in `from_tensor_slices`:

In [ ]:
X = tf.random.uniform([10, 3], dtype=tf.float32)
y = tf.range(10)

ds_joint = tf.data.Dataset.from_tensor_slices((X, y))
for x, t in ds_joint.batch(4):
    print(pd.DataFrame({
        'X1': x.numpy()[:, 0],
        'X2': x.numpy()[:, 1], 
        'X3': x.numpy()[:, 2], 
        'y': t.numpy()
    }), '\n')

## Transformations

Applying transformations to each individual element of a TF dataset is easy &mdash; just call `map`. This will return a dataset where each streamed instance is a transformed version of the original instance.

In [ ]:
X_max = tf.reduce_max(X, axis=0)
X_min = tf.reduce_min(X, axis=0)
ds_transformed = ds_joint.map(lambda x, y: (tf.math.divide(x - X_min, X_max - X_min), y))
for x, t in ds_transformed.batch(4):
    print(pd.DataFrame(
            {
                'X1': x.numpy()[:, 0],
                'X2': x.numpy()[:, 1], 
                'X3': x.numpy()[:, 2], 
                'y': t.numpy()
            }
        ), '\n'
    )

Applying this sort of transformation can be used for a user-defined function. 
For example, if we have a dataset created from the list of image filenames on disk, 
we can define a function to load the images from these filenames and apply that 
function by calling the `.map()` method. 

## Shuffle, Batch, Repeat

To train a neural network using SGD, it is important to feed training data as randomly shuffled batches. Otherwise, the weight updates are biased with regards to ordering of the input data. TensorFlow implements the `.shuffle` method on dataset objects with a `buffer_size` parameter for chunking a dataset that is too large to fit in memory. The tradeoff is that data is only shuffled locally on chunks.

In [ ]:
fig = plt.figure(figsize=(10, 4))
buffer_size = [1, 20, 60, 100]
for i in range(len(buffer_size)):
    shuffled_data = []
    ds_range = tf.data.Dataset.from_tensor_slices(tf.range(100))
    for x in ds_range.shuffle(buffer_size[i]).batch(1):
        shuffled_data.append(x.numpy()[0])

    ax = fig.add_subplot(1, 4, i+1)
    ax.bar(range(100), shuffled_data)
    ax.set_title(f"buffer_size={buffer_size[i]}")

plt.tight_layout()
plt.show()

Larger `buffer_size` results in a more uniform shuffle. Note that `shuffle` has an important argument `reshuffle_each_iteration` that controls whether the shuffle order should be different each time the dataset is iterated over. This is set to `True` by default. 

In [ ]:
dataset = tf.data.Dataset.range(3)
dataset = dataset.shuffle(3)
dataset = dataset.repeat(2)
list(dataset.as_numpy_iterator())

In [ ]:
dataset = tf.data.Dataset.range(3)
dataset = dataset.shuffle(3, reshuffle_each_iteration=False)
dataset = dataset.repeat(2)
list(dataset.as_numpy_iterator())

### Creating epochs

When training a model for multiple epochs, we need to shuffle and iterate over the dataset by the desired number of epochs. To repeat the dataset, we use the `.repeat` method on the dataset object. The following pattern is the correct order of creating epochs. For training, it is recommended to set `drop_remainder=True` in `.batch()` to drop the last mini batch of size 1. 

In [ ]:
buffer_size = 6
for x, t in ds_transformed.shuffle(buffer_size).batch(3).repeat(2):
    print(pd.DataFrame(
            {
                'X1': x.numpy()[:, 0],
                'X2': x.numpy()[:, 1], 
                'X3': x.numpy()[:, 2], 
                'y': t.numpy()
            }
        ), '\n'
    )

To see this more transparently, consider the 1-dimensional dataset:

In [ ]:
data = tf.data.Dataset.range(10).shuffle(6).batch(3).repeat(2)
list(data.as_numpy_iterator())

Note that using `.repeat` requires you to specify `steps_per_epoch` in the Keras `fit` function since we now have a long dataloader (for the reader not familiar with Keras, it will be discusssed later). A way to avoid this is to iterate over epoch numbers, and generating shuffled batches at each iteration. Again the default behavior of reshuffling at each iteration turns out to be very convenient.

In [ ]:
# Define data loader outside training loop
dataset = tf.data.Dataset.range(10)
dataloader = dataset.shuffle(10).batch(3, drop_remainder=True)

# Train loop: iterate over data loader every epoch
num_epochs = 2
for i in range(num_epochs):
    print(f"\n[Epoch {i}]:")
    
    for x in dataloader:
        print("  ", x)

## Dataset from local files

For very large datasets, a good enough approach is to randomly shard the data into multiple files once before training, then shuffle the filenames uniformly. That is, instead of loading actual data into TF datasets, we can simply load references to it in the form of filenames, then we can load the data in batches during training and inference using `.map()`.

```{margin}
`kaggle/v1.5.12`
```

```bash
# Here e.g. https://www.kaggle.com/datasets/waifuai/cat2dog
USER="waifuai"
DATASET="cat2dog"
DATA_DIR=./data
mkdir ${DATA_DIR}
kaggle datasets download -d ${USER}/${DATASET} -p ${DATA_DIR}
unzip ${DATA_DIR}/${DATASET}.zip -d ${DATA_DIR}/${DATASET} > /dev/null
rm ${DATA_DIR}/${DATASET}.zip
```

```text
mkdir: ./data: File exists
Downloading cat2dog.zip to ./data
100%|██████████████████████████████████████| 27.4M/27.4M [00:09<00:00, 3.13MB/s]
```

We can get filenames using `.glob` on a `pathlib.Path` object as follows:

In [ ]:
DATASET_DIR = Path("data").absolute()
cat_imgdir_path = DATASET_DIR / "cat2dog" / "cat2dog" / "trainA"
dog_imgdir_path = DATASET_DIR / "cat2dog" / "cat2dog" / "trainB"

cat_file_list = sorted([str(path) for path in cat_imgdir_path.glob("*.jpg")])
dog_file_list = sorted([str(path) for path in dog_imgdir_path.glob("*.jpg")])

Visualizing image sets for cats and dogs:

In [ ]:
import pathlib

fig, ax = plt.subplots(1, 6, figsize=(10, 5))
for i, file in enumerate(cat_file_list[:6]):
    img_raw = tf.io.read_file(file)
    img = tf.image.decode_image(img_raw)
    ax[i].imshow(img)
    ax[i].set_title(pathlib.Path(file).name, size=12)
    
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=(10, 5))
for i, file in enumerate(dog_file_list[:6]):
    img_raw = tf.io.read_file(file)
    img = tf.image.decode_image(img_raw)
    ax[i].imshow(img)
    ax[i].set_title(pathlib.Path(file).name, size=12)
    
plt.tight_layout()
plt.show()

Instead of having a dataset of arrays for images, and their corresponding labels, we can create a dataset of filenames and their labels. Then, we can transform the filenames to images using a mapping to load and preprocess images given their filenames.

In [ ]:
from functools import partial

# Define mapping function: (filename, label) -> (RGB array, label)
def load_and_preprocess(path, label, img_width=124, img_height=124):
    img_raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_raw, channels=3)
    img = tf.image.resize(img, [img_height, img_width])
    img /= 255.0
    return img, label

# Create dataset of RGB arrays resized to 32x32x3
file_names = cat_file_list + dog_file_list
labels = [0] * len(cat_file_list) + [1] * len(dog_file_list)
file_names_ds = tf.data.Dataset.from_tensor_slices((file_names, labels))
images_dataset = file_names_ds.map(partial(load_and_preprocess, img_width=32, img_height=32))

# Display one image and its label (0 = cat, 1 = dog) 
img, label = next(iter(images_dataset.batch(1)))
print(label.numpy()[0])
plt.figure(figsize=(2, 2))
plt.imshow(img[0, :, :, :]);

## Datasets from `tensorflow_datasets`

The `tensorflow_datasets` library provides a collection of freely available 
(well formatted) datasets for training or evaluating deep learning models which allows for quick 
experimentation. The datasets also come with an `info` dictionary which contains all relevant metadata.
Morevero, the datasets already load as a `Dataset` object. The list of all available datasets can be found in [this catalog](https://www.tensorflow.org/datasets/catalog/overview).

In [ ]:
import tensorflow_datasets as tfds

print(len(tfds.list_builders())) # no. of available datasets
print(tfds.list_builders()[:5])

Datasets from `tfds` can be loaded using three steps:

In [ ]:
coil100_bldr = tfds.builder('coil100')                      
coil100_bldr.download_and_prepare()                         
coil100_ds = coil100_bldr.as_dataset(shuffle_files=True)

import json
json_info = json.loads(coil100_bldr.info.as_json)
print(json.dumps(json_info, indent=2))

We can see that the result is a dictionary:

In [ ]:
print(coil100_ds)

There is only train set in this dataset.

In [ ]:
coil100_ds_trn = coil100_ds['train']
print(coil100_ds_trn)
print(isinstance(coil100_ds_trn, tf.data.Dataset))
print(len(coil100_ds_trn))

In [ ]:
instance = next(iter(coil100_ds_trn))
print(type(instance))
print(instance.keys())

Each element of this dataset is a dictionary, so we have to extract the features and labels using a mapping:

In [ ]:
ds_train = coil100_ds_trn.map(lambda d: (
    {k: d[k] for k in d.keys() if k != 'object_id'},
    d['object_id']
))

# Try one example
features, labels = next(iter(ds_train.batch(8)))
print(features['angle'].shape)
print(features['angle_label'].shape)
print(features['image'].shape)
print(labels.numpy().shape)

In [ ]:
fig = plt.figure(figsize=(8, 3))
for i in range(8):
    img = features['image'][i, :, :, :]
    ax = fig.add_subplot(2, 4, i+1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(img)
    ax.set_title(f"angle={features['angle'][i]}, id={labels[i].numpy()}", size=10)
    
plt.tight_layout()
plt.show()

It turns out that `tfds` has a wrapper function called `load` that performs all the three steps. We will use this to fetch the MNIST dataset in one step:

In [ ]:
MNIST, MNIST_info = tfds.load('mnist', with_info=True, shuffle_files=False)
print(MNIST_info)

In [ ]:
train_dataset, test_dataset = MNIST['train'], MNIST['test']
train_dataset = train_dataset.map(lambda d: (d['image'], d['label']))
test_dataset = test_dataset.map(lambda d: (d['image'], d['label']))

print(type(train_dataset))
print(train_dataset)

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_dataset.batch(8)))
fig = plt.figure(figsize=(8, 3))
for i in range(8):
    img = images[i, :, :, :]
    ax = fig.add_subplot(2, 4, i+1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f"{labels[i].numpy()}", size=15)
    
plt.tight_layout()
plt.show()

## Model training

So far we have learned about the basic utility components of 
TensorFlow for manipulating tensors and organizing data into formats that we 
can iterate over during training. In this section, we look at how to feed data into TensorFlow models.

### From scratch

**Dataset.** In this section, we train a simple linear regression model derived from `tf.keras.Model` by implementing SGD from scratch. The loop iterates over a `Dataset` object which acts as a data loader. We use a 2-layer MLP to learn the artificial dataset composed of 10 points below.

In [ ]:
import numpy as np

X_train = np.arange(10).reshape((10, 1))
y_train = np.array([1.0, 1.3, 3.1, 2.0, 5.0, 6.3, 6.6, 7.4, 8.0, 9.0])

plt.figure(figsize=(7, 6), dpi=80)
plt.grid(linestyle='dashed')
plt.scatter(X_train, y_train, edgecolor="#333", s=50, zorder=3)
plt.xlabel('x')
plt.ylabel('y')
plt.axis('square')
plt.xlim(-1, 10)
plt.ylim(-1, 10)
plt.title('Regression dataset')
plt.show()

In [ ]:
X_train_norm = (X_train - np.mean(X_train)) / np.std(X_train)
ds_train_orig = tf.data.Dataset.from_tensor_slices((
    tf.cast(X_train_norm, tf.float32), 
    tf.cast(y_train, tf.float32)
))

<br>

**Model.** We implement a univariate linear regression model by subclassing the Keras `Model` class.

In [ ]:
class RegressionModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.w = tf.Variable(0.0, name='weight')
        self.b = tf.Variable(0.0, name='bias')
    
    def call(self, x):
        return self.w * x + self.b

<br>

**Training loop.** The `train` function implements a single step of SGD optimization where gradients of the MSE loss function obtained automatically are used to update the weight `w` and bias `b`. Note that using `count=None` on `repeat` will create a batched version of the dataset that repeats infinitely many times. But since we implement no early stopping mechanism, we set `count=200` to train the model for 200 epochs. 

In [ ]:
def loss_fn(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

@tf.function
def train(model, inputs, outputs, learning_rate):
    with tf.GradientTape() as tape:
        loss = loss_fn(model(inputs), outputs)
    
    dw, db = tape.gradient(loss, [model.w, model.b])
    model.w.assign_sub(learning_rate * dw)
    model.b.assign_sub(learning_rate * db)

Alternatively, we can exploit the `apply_gradients` method of built-in Keras optimizers:

```python
optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
optimizer.apply_gradients(zip([dw, db], [model.w, model.b]))
```

Finally, we can implement the train loop by iterating over the batch loader and applying the train step at each iteration:

In [ ]:
# Hyperparameters
NUM_EPOCHS = 200
LEARNING_RATE = 0.001
BATCH_SIZE = 1

# Instantiate model
model = RegressionModel()

# Create batch loader
ds_train = ds_train_orig.shuffle(buffer_size=len(y_train))
ds_train = ds_train.batch(1)
ds_train = ds_train.repeat(count=NUM_EPOCHS)

ws, bs = [], []
steps_per_epoch = np.floor(len(y_train) / BATCH_SIZE)
for i, batch in enumerate(ds_train):
    ws.append(model.w.numpy())
    bs.append(model.b.numpy())

    bx, by = batch
    loss_value = loss_fn(model(bx), by)
    train(model, bx, by, learning_rate=LEARNING_RATE)
    
    if i%100==0:
        print(f'Epoch {i // int(steps_per_epoch):4d} Loss {loss_value:>8.4f}')


<br>

**History.** Here we plot the learned model and the history of its parameters. As training progresses, the weight and bias converge to an optimal value.

In [ ]:
print(f'Final Parameters: w={model.w.numpy():.4f}, b={model.b.numpy():.4f}')

# Generate test set; here test = inference
X_test = np.linspace(0, 9, num=100).reshape(-1, 1)
X_test_norm = (X_test - np.mean(X_train)) / np.std(X_train)

# Get predictions on test set
y_pred = model(tf.cast(X_test_norm, dtype=tf.float32))

# Plot learned model
fig = plt.figure(figsize=(13, 5), dpi=600)
ax = fig.add_subplot(1, 2, 1)
plt.scatter(X_train_norm, y_train, c="#1F77B4", edgecolor="#333", zorder=3)
plt.plot(X_test_norm, y_pred, linestyle="--", color='black', lw=2, zorder=3)
plt.legend(['train data', 'model'], fontsize=12)
ax.set_xlabel('x', size=12)
ax.set_ylabel('y', size=12)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.grid(linestyle='dashed', alpha=0.8)

# Plot parameter history
ax = fig.add_subplot(1, 2, 2)
plt.plot(ws, lw=3)
plt.plot(bs, lw=3)
plt.legend([r'weight', r'bias'], fontsize=12)
ax.set_xlabel('step', size=12)
ax.set_ylabel('value', size=12)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.grid(linestyle='dashed', alpha=0.8)

plt.show()

### Keras `fit` function

In this section, we look at how to obtain a dataset from `tensorflow_datasets` and use it to train a Keras model. In particular we will use the Iris Dataset which consists of 150 observations of the petal and sepal lengths, and petal and sepal widths of 3 different types of irises.


<!-- ```{figure} ../../img/iris.jpeg
---
name: iris
---
From left to right: [Iris setosa](https://commons.wikimedia.org/w/index.php?curid=170298), [Iris versicolor](https://commons.wikimedia.org/w/index.php?curid=248095), and [Iris virginica](https://www.flickr.com/photos/33397993@N05/3352169862).
``` -->

In [ ]:
iris, iris_info = tfds.load('iris', with_info=True)
print(iris_info.splits) 

In [ ]:
type(iris['train'])

<br>

**Dataset split.** This only has a train set, so we have to manually split for validation. We can do this with the `.take()` and `.skip()` methods. But this can lead to some unexpected behavior after calling `.shuffle` which converts the dataset to a `ShuffleDataset` which would shuffle the after the initial application of take when creating the train dataset. A workaround is to set `reshuffle_each_iteration` to `False`. 

In [ ]:
tf.random.set_seed(1)

# Shuffle data
dataset_orig = iris['train']
N = len(dataset_orig)
dataset_shuffled = dataset_orig.shuffle(N, reshuffle_each_iteration=False)

# Split into train and test sets + transform
train_dataset = dataset_shuffled.take(100)
test_dataset  = dataset_shuffled.skip(100)
print("Train size:", len(train_dataset))
print("Test size: ", len(test_dataset))

train_dataset = train_dataset.map(lambda d: (d['features'], d['label']))
test_dataset  =  test_dataset.map(lambda d: (d['features'], d['label']))

<br>

**Model.** A two-layer MLP with sigmoid activations should suffice to learn 100 data points:

In [ ]:
iris_model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='sigmoid', name='fc1', input_shape=(4,)),
    tf.keras.layers.Dense(3, name='fc2', activation='softmax')
])

iris_model.summary() # No need to call .build(), input_shape passed in first dense layer.

<br>

**Training.** Observe that the Keras `fit` method works with `tf.data` batch loaders:

In [ ]:
# Use sparse since targets are 0, 1, 2
iris_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.1), 
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Hyperparameters
NUM_EPOCHS = 50
BATCH_SIZE = 8

# This will be iterated over in the fit method.
train_loader = train_dataset.shuffle(buffer_size=100)
train_loader = train_loader.batch(batch_size=BATCH_SIZE, drop_remainder=True)
train_loader = train_loader.prefetch(buffer_size=10) # Prepare next elements 

# Train model
history = iris_model.fit(
    train_loader, 
    epochs=NUM_EPOCHS,
    verbose=1,
)

In [ ]:
hist = history.history

# Train loss plot
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(hist['loss'], lw=2, label='train')
ax[0].set_title('Loss', size=15)
ax[0].set_xlabel('Epoch', size=15)
ax[0].tick_params(axis='both', which='major', labelsize=15)
ax[0].grid(linestyle='dashed')
ax[0].legend(fontsize=15)

# Accuracy plot
ax[1].plot(hist['accuracy'], lw=2, label='train')
ax[1].set_title('Accuracy', size=15)
ax[1].set_xlabel('Epoch', size=15)
ax[1].tick_params(axis='both', which='major', labelsize=15)
ax[1].grid(linestyle='dashed')
ax[1].legend(fontsize=15)

plt.tight_layout()

<br>

**Evaluation.** Keras methods `evaluate` and `predict` work nicely with TF dataset objects:

In [ ]:
results = iris_model.evaluate(test_dataset.batch(1), verbose=0) # 50 batches of size 1
print('Test loss: {:.4f}   Test Acc.: {:.4f}'.format(*results))